# 02d - Stage 1 Combined Forecast Charts

Reads Stage1_final_regressors_US_Q.csv (the output of 02c) and produces
two figures inline:

- Figure A: Stationary transforms for all 12 winning forecasts,
  with 10 years of actual history for context.
- Figure B: Back-transformed levels for all 12 variables.

Formatting matches 02b Figures 7.1 and 7.2.
Teal = DM-validated winner, Amber = unvalidated winner.

## Section 0 - Imports and configuration

In [18]:
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Data source - raw GitHub URL matching 02c convention
GITHUB_RAW      = 'https://raw.githubusercontent.com/hogandan85/ST-498/refs/heads/main/Data%20Collection'
REGRESSOR_FILE  = f'{GITHUB_RAW}/Stage1_final_regressors_US_Q.csv'

# Time boundaries
FORECAST_START  = pd.Timestamp('2026-03-31')
FORECAST_END    = pd.Timestamp('2030-12-31')
COVID_START     = pd.Timestamp('2020-03-31')
COVID_END       = pd.Timestamp('2021-12-31')

# Number of historical quarters to show before the forecast start (40 = 10 years)
HISTORY_QUARTERS = 40

# Colours matching 02b exactly
NAVY    = '#1F3864'
TEAL    = '#17A589'
AMBER   = '#E67E22'
GREY    = '#7F8C8D'
TEMPLATE = 'plotly_white'

## Section 1 - Winner selection metadata

Hardcoded from the 02c selection DataFrame output for reproducibility.
Update here if 02c is re-run and selections change.

In [19]:
# Results from 02c winner selection
WINNER_INFO = {
    'us_bond_yield_10y':     {'winner': 'ML',  'model': 'LASSO',       'validated': True},
    'us_consumer_confidence':{'winner': 'TS',  'model': 'AR',          'validated': False},
    'us_cpi':                {'winner': 'ML',  'model': 'KRR',         'validated': True},
    'us_credit_qoq_growth':  {'winner': 'ML',  'model': 'XGBoost',     'validated': False},
    'us_gdp_yoy_growth':     {'winner': 'ML',  'model': 'XGBoost',     'validated': True},
    'us_house_price_yoy':    {'winner': 'ML',  'model': 'Ridge',       'validated': True},
    'us_indprod_yoy':        {'winner': 'TS',  'model': 'SARMA',       'validated': False},
    'us_oil_yoy':            {'winner': 'ML',  'model': 'Elastic Net', 'validated': True},
    'us_reer_diff':          {'winner': 'ML',  'model': 'KRR',         'validated': False},
    'us_sp500_log_ret':      {'winner': 'ML',  'model': 'Ridge',       'validated': False},
    'us_unemployment':       {'winner': 'TS',  'model': 'SARMA',       'validated': True},
    'us_vix_log_ret':        {'winner': 'ML',  'model': 'SVR',         'validated': False},
}

# Maps each stationary variable to its level column in the combined CSV
# For the 4 direct-level variables (bond yield, CPI, consumer confidence, unemployment)
# the level column is the same as the stationary column - Figure B will look
# identical to Figure A for those panels, which is correct.
LEVEL_COLS = {
    'us_bond_yield_10y':     'us_bond_yield_10y',
    'us_consumer_confidence':'us_consumer_confidence',
    'us_cpi':                'us_cpi',
    'us_credit_qoq_growth':  'us_credit',
    'us_gdp_yoy_growth':     'us_real_gdp',
    'us_house_price_yoy':    'us_house_price_idx',
    'us_indprod_yoy':        'us_industrial_production',
    'us_oil_yoy':            'us_oil_price',
    'us_reer_diff':          'us_reer',
    'us_sp500_log_ret':      'us_sp500_close',
    'us_unemployment':       'us_unemployment',
    'us_vix_log_ret':        'us_vix',
}

# Alphabetical order, matching Table 6.X in the report
VARIABLES = sorted(WINNER_INFO.keys())

N_COLS = 3
N_ROWS = int(np.ceil(len(VARIABLES) / N_COLS))

## Section 2 - Load data

In [20]:
df = pd.read_csv(REGRESSOR_FILE, index_col=0, parse_dates=True)
df = df.sort_index()

history  = df[df.index <  FORECAST_START].copy()
forecast = df[df.index >= FORECAST_START].copy()
chart_history_start = history.index[-HISTORY_QUARTERS]

print(f'Loaded:         {df.shape}  ({df.index.min().date()} to {df.index.max().date()})')
print(f'History window: {chart_history_start.date()} to {history.index[-1].date()}')
print(f'Forecast:       {forecast.index[0].date()} to {forecast.index[-1].date()}')

Loaded:         (164, 42)  (1990-03-31 to 2030-12-31)
History window: 2016-03-31 to 2025-12-31
Forecast:       2026-03-31 to 2030-12-31


## Section 3 - Figure A: Stationary transforms

Matches 02b Figure 7.1 formatting. Shows 10 years of actual history plus
the 20-quarter combined winner forecast per variable.
Subplot title format: variable name with winning model and validation status.

In [21]:
# Subplot titles matching 02b Figure 7.1 format exactly
titles_a = [
    f"{v}  ({WINNER_INFO[v]['model']}{'' if WINNER_INFO[v]['validated'] else ' - unvalidated'})"
    for v in VARIABLES
]

fig_a = make_subplots(
    rows=N_ROWS, cols=N_COLS,
    subplot_titles=titles_a,
    vertical_spacing=0.09,
    horizontal_spacing=0.06
)

for i, var in enumerate(VARIABLES):
    row, col   = i // N_COLS + 1, i % N_COLS + 1
    validated  = WINNER_INFO[var]['validated']
    line_color = TEAL if validated else AMBER

    h = history.loc[chart_history_start:, var].dropna()
    f = forecast[var].dropna()

    # History trace
    fig_a.add_trace(
        go.Scatter(x=h.index, y=h.values, mode='lines',
                   line=dict(color=NAVY, width=1.8),
                   showlegend=(i == 0), name='History'),
        row=row, col=col
    )
    # Forecast trace
    fig_a.add_trace(
        go.Scatter(x=f.index, y=f.values, mode='lines',
                   line=dict(color=line_color, width=2),
                   showlegend=(i == 0), name='Forecast'),
        row=row, col=col
    )
    # Vertical line at forecast start
    fig_a.add_vline(
        x=h.index.max(), line_dash='dot', line_color=GREY, line_width=1,
        row=row, col=col
    )

fig_a.update_layout(
    title=dict(
        text='Figure A - Stage 1 Combined Winner Selection: Stationary Transforms '
             '(teal = validated, amber = unvalidated winner)',
        font=dict(size=13, color=NAVY)
    ),
    template=TEMPLATE,
    height=280 * N_ROWS,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(t=110, b=30, l=40, r=20)
)
fig_a.update_annotations(font_size=10)
fig_a.show()

## Section 4 - Figure B: Back-transformed levels

Matches 02b Figure 7.2 formatting. Level columns are read directly from the
combined CSV, where 02c reconstructed them in Section 3b.
Subplot title format matches 02b: level column name with the stationary transform
variable in parentheses.

In [22]:
# Subplot titles matching 02b Figure 7.2 format exactly:
# "{raw_col}  (transform - {var_name})"
titles_b = [
    f"{LEVEL_COLS[v]}  (transform - {v})"
    for v in VARIABLES
]

fig_b = make_subplots(
    rows=N_ROWS, cols=N_COLS,
    subplot_titles=titles_b,
    vertical_spacing=0.09,
    horizontal_spacing=0.06
)

for i, var in enumerate(VARIABLES):
    row, col   = i // N_COLS + 1, i % N_COLS + 1
    level_col  = LEVEL_COLS[var]
    validated  = WINNER_INFO[var]['validated']
    line_color = TEAL if validated else AMBER

    mask_hist = (df.index >= chart_history_start) & (df.index < FORECAST_START)
    h = df.loc[mask_hist, level_col].dropna()
    f = df.loc[df.index >= FORECAST_START, level_col].dropna()

    if h.empty:
        continue

    # History trace (level)
    fig_b.add_trace(
        go.Scatter(x=h.index, y=h.values, mode='lines',
                   line=dict(color=NAVY, width=1.8),
                   showlegend=(i == 0), name='History (level)'),
        row=row, col=col
    )
    # Forecast trace (level)
    fig_b.add_trace(
        go.Scatter(x=f.index, y=f.values, mode='lines',
                   line=dict(color=line_color, width=2),
                   showlegend=(i == 0), name='Forecast (level)'),
        row=row, col=col
    )
    # Vertical line at forecast start
    fig_b.add_vline(
        x=h.index.max(), line_dash='dot', line_color=GREY, line_width=1,
        row=row, col=col
    )

fig_b.update_layout(
    title=dict(
        text='Figure B - Stage 1 Combined Winner Selection: Back-Transformed Levels '
             '(teal = validated, amber = unvalidated winner)',
        font=dict(size=13, color=NAVY)
    ),
    template=TEMPLATE,
    height=280 * N_ROWS,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(t=110, b=30, l=40, r=20)
)
fig_b.update_annotations(font_size=10)
fig_b.show()